<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week4/week4_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1 — Week 4 Header & Library Imports
# Eye of Ra 👁️ — Week 4: Streamlit Dashboard & Technical Abstract

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Libraries loaded — Week 4 begins")
print("👁️ Eye of Ra — Dashboard & Submission")

# Git: Cell 1 — Week 4 header and imports

✅ Libraries loaded — Week 4 begins
👁️ Eye of Ra — Dashboard & Submission


In [3]:
# Cell 2 — Load dataset and scored contracts
df = pd.read_csv('/content/eye_of_ra_master_dataset_v3.csv')
scored = pd.read_csv('/content/eye_of_ra_scored_contracts_v1.csv')
model = joblib.load('/content/eye_of_ra_model_v1.pkl')
feature_cols = joblib.load('/content/eye_of_ra_features_v1.pkl')

print(f"✅ Master dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"✅ Scored contracts: {scored.shape[0]} rows x {scored.shape[1]} columns")
print(f"✅ Model loaded: {type(model).__name__}")
print(f"✅ Features loaded: {len(feature_cols)} features")
print(f"\nTier breakdown:")
print(scored['tier'].value_counts())

# Git: Cell 2 — load all files for Week 4

✅ Master dataset: 262 rows x 21 columns
✅ Scored contracts: 262 rows x 10 columns
✅ Model loaded: XGBClassifier
✅ Features loaded: 12 features

Tier breakdown:
tier
🟢 MONITOR     240
🔴 ESCALATE     17
🟡 REVIEW        5
Name: count, dtype: int64


In [4]:
# Cell 3 — Install Streamlit and ngrok
!pip install streamlit pyngrok -q

from pyngrok import ngrok

print("✅ Streamlit installed")
print("✅ pyngrok installed")

# Git: Cell 3 — install Streamlit and ngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.5 MB/s eta 0:00:00
✅ Streamlit installed
✅ pyngrok installed


In [5]:
# Cell 4 — Write the Streamlit dashboard file

dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Page config
st.set_page_config(
    page_title="Eye of Ra 👁️",
    page_icon="👁️",
    layout="wide"
)

# Load data
@st.cache_data
def load_data():
    scored = pd.read_csv("/content/eye_of_ra_scored_contracts_v1.csv")
    return scored

scored = load_data()

# Clean up flags column for display
scored["flags"] = scored["flags"].fillna("[]")

# ── SIDEBAR ──────────────────────────────────────────────
st.sidebar.image("https://flagcdn.com/w80/gh.png", width=60)
st.sidebar.title("👁️ Eye of Ra")
st.sidebar.markdown("AI Procurement Fraud Detection for Ghana")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", [
    "📊 Overview",
    "🔴 Flagged Contracts",
    "🏛️ Entity Scorecards",
    "🔍 Contract Deep Dive"
])
st.sidebar.markdown("---")
st.sidebar.markdown("Built on Ghana PPA & Auditor-General data")
st.sidebar.markdown("Grounded in Public Procurement Act 663")

# ── PAGE 1: OVERVIEW ─────────────────────────────────────
if page == "📊 Overview":
    st.title("👁️ Eye of Ra — Procurement Fraud Detection")
    st.markdown("### Ghana AI Summit 2026 | Real Data. Real Flags. Real Accountability.")
    st.markdown("---")

    # Top metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Contracts Scanned", "262")
    col2.metric("Fraud Detected", "87%", "20 of 23 confirmed")
    col3.metric("🔴 Escalate", str(len(scored[scored["tier"] == "🔴 ESCALATE"])))
    col4.metric("False Positives", "0", "in ESCALATE tier")

    st.markdown("---")

    col1, col2 = st.columns(2)

    # Tier distribution pie
    with col1:
        tier_counts = scored["tier"].value_counts()
        fig = px.pie(
            values=tier_counts.values,
            names=tier_counts.index,
            title="Risk Tier Distribution — All 262 Contracts",
            color=tier_counts.index,
            color_discrete_map={
                "🔴 ESCALATE": "#d32f2f",
                "🟡 REVIEW": "#f9a825",
                "🟢 MONITOR": "#388e3c"
            }
        )
        st.plotly_chart(fig, use_container_width=True)

    # Fraud detection bar
    with col2:
        fig2 = go.Figure(go.Bar(
            x=["🔴 ESCALATE\\n(17 caught)", "🟡 REVIEW\\n(3 caught)", "🟢 MONITOR\\n(3 missed)"],
            y=[17, 3, 3],
            marker_color=["#d32f2f", "#f9a825", "#388e3c"],
            text=[17, 3, 3],
            textposition="outside"
        ))
        fig2.update_layout(
            title="23 Confirmed Fraud Contracts — Detection Breakdown",
            yaxis_title="Number of Contracts",
            showlegend=False
        )
        st.plotly_chart(fig2, use_container_width=True)

    # Score distribution
    st.markdown("### Score Distribution — Fraud vs Clean")
    fig3 = px.histogram(
        scored, x="composite_score", color="fraud_label",
        nbins=30,
        color_discrete_map={0: "#388e3c", 1: "#d32f2f"},
        labels={"fraud_label": "Fraud", "composite_score": "Composite Risk Score"},
        title="Risk Score Distribution"
    )
    fig3.add_vline(x=50, line_dash="dash", line_color="#d32f2f", annotation_text="ESCALATE threshold")
    fig3.add_vline(x=35, line_dash="dash", line_color="#f9a825", annotation_text="REVIEW threshold")
    st.plotly_chart(fig3, use_container_width=True)

# ── PAGE 2: FLAGGED CONTRACTS ─────────────────────────────
elif page == "🔴 Flagged Contracts":
    st.title("🔴 Flagged Contracts")
    st.markdown("Contracts in ESCALATE and REVIEW tiers — sorted by risk score.")
    st.markdown("---")

    tier_filter = st.multiselect(
        "Filter by tier:",
        options=["🔴 ESCALATE", "🟡 REVIEW", "🟢 MONITOR"],
        default=["🔴 ESCALATE", "🟡 REVIEW"]
    )

    filtered = scored[scored["tier"].isin(tier_filter)].sort_values("composite_score", ascending=False)

    st.markdown(f"**{len(filtered)} contracts shown**")
    st.dataframe(
        filtered[["entity", "supplier", "composite_score", "tier", "fraud_label"]].reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 3: ENTITY SCORECARDS ─────────────────────────────
elif page == "🏛️ Entity Scorecards":
    st.title("🏛️ Entity Integrity Scorecards")
    st.markdown("Search any procurement entity and see their full risk profile.")
    st.markdown("---")

    entity_list = sorted(scored["entity"].unique())
    selected_entity = st.selectbox("Select an entity:", entity_list)

    entity_contracts = scored[scored["entity"] == selected_entity]
    max_score = entity_contracts["composite_score"].max()
    fraud_count = entity_contracts["fraud_label"].sum()
    total = len(entity_contracts)
    escalate_count = len(entity_contracts[entity_contracts["tier"] == "🔴 ESCALATE"])

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Contracts", total)
    col2.metric("Max Risk Score", f"{max_score:.1f}")
    col3.metric("Confirmed Fraud", int(fraud_count))
    col4.metric("Escalated", escalate_count)

    st.markdown("---")
    st.markdown("### All contracts for this entity:")
    st.dataframe(
        entity_contracts[["supplier", "composite_score", "tier", "fraud_label"]].sort_values("composite_score", ascending=False).reset_index(drop=True),
        use_container_width=True
    )

# ── PAGE 4: CONTRACT DEEP DIVE ────────────────────────────
elif page == "🔍 Contract Deep Dive":
    st.title("🔍 Contract Deep Dive")
    st.markdown("Select any contract for a full risk breakdown.")
    st.markdown("---")

    contract_idx = st.selectbox(
        "Select contract index:",
        options=scored.index.tolist(),
        format_func=lambda i: f"#{i} — {scored.loc[i, 'entity'][:50]}"
    )

    row = scored.loc[contract_idx]

    col1, col2 = st.columns(2)
    with col1:
        st.markdown(f"**Entity:** {row['entity']}")
        st.markdown(f"**Supplier:** {row['supplier']}")
        st.markdown(f"**Composite Score:** {row['composite_score']}")
        st.markdown(f"**Tier:** {row['tier']}")
        st.markdown(f"**Fraud Confirmed:** {'✅ Yes' if row['fraud_label'] == 1 else '⬜ No'}")

    with col2:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=row["composite_score"],
            title={"text": "Risk Score"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"color": "#d32f2f" if row["composite_score"] >= 50 else "#f9a825" if row["composite_score"] >= 35 else "#388e3c"},
                "steps": [
                    {"range": [0, 35], "color": "#e8f5e9"},
                    {"range": [35, 50], "color": "#fff9c4"},
                    {"range": [50, 100], "color": "#ffebee"}
                ],
                "threshold": {"line": {"color": "black", "width": 4}, "thickness": 0.75, "value": row["composite_score"]}
            }
        ))
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")
    st.markdown("### Flags raised:")
    flags = eval(row["flags"]) if isinstance(row["flags"], str) else row["flags"]
    if flags:
        for f in flags:
            st.markdown(f"• {f}")
    else:
        st.markdown("No flags raised.")

    st.markdown("### Legal citations:")
    citations = eval(row["legal_citations"]) if isinstance(row["legal_citations"], str) else row["legal_citations"]
    if citations:
        for c in citations:
            st.markdown(f"• {c}")

    st.markdown("### Top SHAP drivers:")
    shap_items = eval(row["shap_top3"]) if isinstance(row["shap_top3"], str) else row["shap_top3"]
    if shap_items:
        for s in shap_items:
            st.markdown(f"• {s}")
'''

with open("/content/app.py", "w") as f:
    f.write(dashboard_code)

print("✅ Dashboard file written: /content/app.py")

# Git: Cell 4 — Streamlit dashboard file written

✅ Dashboard file written: /content/app.py


In [6]:
# Cell 5 — Launch the Streamlit dashboard
import subprocess
import threading
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()

# Start Streamlit in background
def run_streamlit():
    subprocess.run(["streamlit", "run", "/content/app.py", "--server.port", "8501", "--server.headless", "true"])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to start
import time
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print("=" * 55)
print("👁️  EYE OF RA DASHBOARD IS LIVE")
print("=" * 55)
print(f"\n🌐 Open this URL in your browser:")
print(f"   {public_url}")
print(f"\nKeep this cell running — closing it kills the dashboard.")

# Git: Cell 5 — dashboard launched

ERROR:pyngrok.process.ngrok:t=2026-06-06T00:55:16+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-06T00:55:16+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.